# Cross-dataset comparison

Loads **JSON summaries** written by the per-dataset EDA notebooks (`data/processed/eda_summaries/*.json`). Run those notebooks first, or summaries will be missing.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

_HERE = Path.cwd().resolve()
if (_HERE / "data" / "datasets").is_dir():
    PROJECT_ROOT = _HERE
elif (_HERE.parent / "data" / "datasets").is_dir():
    PROJECT_ROOT = _HERE.parent
else:
    raise FileNotFoundError("Run from repo root or notebooks/")

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import eda_helpers as edh

pd.set_option("display.max_colwidth", 100)

try:
    from IPython.display import display
except ImportError:
    display = print

print("PROJECT_ROOT:", PROJECT_ROOT)


## Load summaries

In [ ]:
slugs = ["twitter_brand", "sentiment140", "reddit_ml_ready"]
slug_to_notebook = {
    "twitter_brand": "eda_twitter_sentiment.ipynb",
    "sentiment140": "eda_sentiment140.ipynb",
    "reddit_ml_ready": "eda_reddit_comments.ipynb",
}
rows = []
for s in slugs:
    data = edh.load_eda_summary(s, PROJECT_ROOT)
    if data is None:
        print("MISSING summary:", s, "| run", slug_to_notebook.get(s, s))
        continue
    if data.get("dataset") == "twitter_brand":
        row_count = f"train {data['row_count_train']} / test {data['row_count_test']}"
    else:
        row_count = data.get("row_count")
    lab = data.get("label_space")
    lab_s = str(lab) if lab is not None else ""
    rows.append({
        "dataset": data.get("dataset"),
        "row_count": row_count,
        "text_column": data.get("text_column"),
        "label_column": data.get("label_column"),
        "label_space": (lab_s[:80] + "...") if len(lab_s) > 80 else lab,
        "avg_word_count": data.get("avg_word_count"),
        "median_word_count": data.get("median_word_count"),
        "duplicate_text_pct": data.get("duplicate_text_pct"),
        "null_pct": data.get("null_pct"),
        "preprocessing": data.get("preprocessing"),
    })


## Comparison table

In [ ]:
if not rows:
    print("No summaries found. Run the three dataset EDA notebooks first.")
else:
    cmp_df = edh.build_comparison_table(rows)
    display(cmp_df)
    out = PROJECT_ROOT / "data" / "processed" / "eda_dataset_comparison.csv"
    cmp_df.to_csv(out, index=False)
    print("Saved:", out)


## Recommendation (unified training)

- **Text + labels**: prioritize **Sentiment140** (after remap), **Reddit** (after label check), **Twitter train** (after string remap).
- **Align schema**: `text`, `label_int` in {0,1,2}, `source` column.
- **Test Twitter CSV**: no labels — hold out for inference benchmarks only.
- **Scale**: subsample large sources; stratify by source and class.

In [ ]:
print("Action: run eda_twitter_sentiment, eda_sentiment140, eda_reddit_comments, then re-run this notebook to refresh the table.")
